### Augmented Data Preparation

This code will prepare the augmented data for evaluation purposes. This will take the random images from the collected data, augment them, and then put same set of open-ended questions for all the augmentations. The purpose of this evaluation is to ensure that same set of questions/answers with multiple augmented images make complete sense

https://github.com/DIAGNijmegen/pathology-he-auto-augment \
https://www.sciencedirect.com/science/article/pii/S0010482524001021?via%3Dihub \
https://github.com/DIAGNijmegen/pathology-he-autoaugmetation

In [ ]:
import os
import sys

In [ ]:
# Append the directory to the system path
sys.path.append(os.path.join(os.getcwd(), "pathology-he-autoaugmetation/randaugment/train"))

In [ ]:
import pandas as pd
import numpy as np
from randaugment_new_ranges import distort_image_with_randaugment
from PIL import Image
from tqdm import tqdm
import re
import random

#### Reading the original data

In [ ]:
vqa_data = pd.read_csv('data/path_open_vqa.csv')
vqa_data = vqa_data.rename(columns={vqa_data.columns[2]: 'Image1_Path',
                                    vqa_data.columns[26]: 'Image2_Path',
                                    vqa_data.columns[27]: 'Image3_Path',
                                    vqa_data.columns[28]: 'Image4_Path',
                                    vqa_data.columns[29]: 'Image5_Path',
                                    'Image 1 Magnification ': 'Image1_Mag',
                                    'Image 2 Magnification ': 'Image2_Mag',
                                    'Image 3 Magnification ': 'Image3_Mag',
                                    'Image 4 Magnification ': 'Image4_Mag',
                                    'Image 5 Magnification ': 'Image5_Mag'})

vqa_data = vqa_data.reset_index()
vqa_data['index'] = vqa_data['index'] + 1
vqa_data = vqa_data.rename(columns={'index': 'Case_ID'})
vqa_data.head()

Getting only limited columns for the image augmentations

In [ ]:
vqa_data_filt = vqa_data[['Case_ID', 'Pathologist ID', 'Image1_Path', 'Image2_Path', 'Image3_Path']]
vqa_data_filt.head()

Separating the multiple image paths available for the same question to form multiple questions

In [ ]:
vqa_data_filt_final = pd.DataFrame()
for index, row in vqa_data_filt.iterrows():
    # Get the first image
    image_path = row['Image1_Path']
    if pd.notnull(image_path):
        row['Image_Path'] = image_path
        row['Image_ID'] = 'img_pathopen_' + str(row['Case_ID']) + '_01'
        vqa_data_filt_final = pd.concat([vqa_data_filt_final, row.to_frame().T], ignore_index=True)
    
    # Get the second image
    image_path = row['Image2_Path']
    if pd.notnull(image_path):
        row['Image_Path'] = image_path
        row['Image_ID'] = 'img_pathopen_' + str(row['Case_ID']) + '_02'
        vqa_data_filt_final = pd.concat([vqa_data_filt_final, row.to_frame().T], ignore_index=True)
    
    # Get the third image
    image_path = row['Image3_Path']
    if pd.notnull(image_path):
        row['Image_Path'] = image_path
        row['Image_ID'] = 'img_pathopen_' + str(row['Case_ID']) + '_03'
        vqa_data_filt_final = pd.concat([vqa_data_filt_final, row.to_frame().T], ignore_index=True)

vqa_data_filt_final.head()

Removing the un-necessary columns

In [ ]:
vqa_data_filt_final = vqa_data_filt_final.drop(columns=['Image1_Path', 'Image2_Path', 'Image3_Path'])
vqa_data_filt_final.head()

### Generating the Google Sheet Containing the mapping between Google Drive Links and Image Names

This will help to map the image names downloaded from collected data with the Google Drive Links we have in our data

In [ ]:
# function listFilesInNestedFolders(parentFolderId) {
#   // Get the initial parent folder
#   var parentfolder_id = '1VdFeRx06PrTTCbeMVQ0wQ8ML8QF8sSxM-QWlTcMx7TKQAjpWRuW5-EWJi-DtuBDPeXTbGYNm'
#   const parentFolder = DriveApp.getFolderById(parentfolder_id);
#   var folderlisting = 'File Names and Links - '+ parentfolder_id;

#   var ss = SpreadsheetApp.create(folderlisting);
#   var sheet = ss.getActiveSheet();
#   sheet.appendRow(['name','link']);

#   // Get subfolders in the current folder and process them recursively
#   var subfolders = parentFolder.getFolders();
#   while (subfolders.hasNext()) {
#     var subfolder = subfolders.next();
#     // Get files in the current folder
#     var contents = subfolder.getFiles();
#     var file;
#     var name;
#     var link;
#     while (contents.hasNext()) {
#       file = contents.next();
#       name = file.getName();
#       link = file.getUrl();
#       sheet.appendRow([name,link]);
#     }
#   }
# }

Now Download the file generated which contains mapping between google drive links and names.

In [ ]:
data_dir = "data_eval"
file_name = "file_names_and_links_PathOpen_Images.csv"
file_path = os.path.join(data_dir, file_name)
google_drive_name_links = pd.read_csv(file_path)
google_drive_name_links.head()

Extracting the ID's from image links

In [ ]:
vqa_data_filt_final['google_drive_image_id'] = vqa_data_filt_final['Image_Path'].apply(lambda x: x.split('=')[-1])
google_drive_name_links['google_drive_image_id'] = google_drive_name_links['link'].apply(lambda x: x.split('/')[-1])

Combining the names of the image files

In [ ]:
vqa_data_filt_final = pd.merge(vqa_data_filt_final, google_drive_name_links[['google_drive_image_id', 'name']], on='google_drive_image_id', how='left')
vqa_data_filt_final.head()

Creating a dictionary between Image_ID and name

In [ ]:
name_imageid_dict = dict(zip(vqa_data_filt_final['name'], vqa_data_filt_final['Image_ID']))

In [ ]:
name_imageid_dict

### Reading the dataset files

Preparing the augmented images for each filename

In [ ]:
source_dataset_dir = "PathOPEN_Images"
destination_dataset_dir = "PathOPEN_Images_Augmented"
os.makedirs(destination_dataset_dir, exist_ok=True)
pathopen_file_names = os.listdir(source_dataset_dir)

Now peform the augmentation on these images

In [ ]:
TOTAL_AUGMENTAIONS = 30  # Number of augmentations per image
for i in range(len(pathopen_file_names)):
    src_image = pathopen_file_names[i]
    src_image_path = os.path.join(source_dataset_dir, src_image)
    src_image_id = name_imageid_dict[src_image]
    dst_image_aug_folder = os.path.join(destination_dataset_dir, src_image_id + "_augmented")
    os.makedirs(dst_image_aug_folder, exist_ok=True)

    pil_image = Image.open(src_image_path).convert("RGB")
    np_array_img = np.array(pil_image)
    for j in tqdm(range(TOTAL_AUGMENTAIONS), desc=f"[{i+1}/{len(pathopen_file_names)}] Augmenting {src_image_id}"):
        # Apply RandAugment with specified parameters
        conv_img_arr = distort_image_with_randaugment(np_array_img, num_layers=5, magnitude=5, randomize=True, randaugment_transforms_set='custom')
        conv_img = Image.fromarray(conv_img_arr)
        conv_img.save(os.path.join(dst_image_aug_folder, src_image_id + f"_aug_{j}.png"))